In [0]:
%pip install langchain databricks-langchain databricks-sdk langchain-community
%restart_python

In [0]:
import random
from typing import Union
from langchain.schema import HumanMessage, AIMessage
from langchain.chat_models import ChatDatabricks
from langchain.schema.output_parser import StrOutputParser
from langchain.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableBranch, RunnableLambda
from databricks_langchain.genie import GenieAgent
import os

In [0]:

class ConditionalChatbot:
    def __init__(self, databricks_host: str = None, databricks_token: str = None, genie_space_id: str = None):
        """
        Initialize the conditional chatbot.
        
        Args:
            databricks_host: Databricks workspace URL
            databricks_token: Databricks access token
            genie_space_id: Genie space ID for enhanced context
        """
        self.databricks_host = databricks_host or os.getenv("DATABRICKS_HOST")
        self.databricks_token = databricks_token or os.getenv("DATABRICKS_TOKEN")
        self.genie_space_id = genie_space_id or os.getenv("GENIE_SPACE_ID")
        self.llm = ChatDatabricks(
            endpoint="databricks-gpt-oss-20b",
            host=self.databricks_host,
            token=self.databricks_token
        )
        self.genie_agent = GenieAgent(
            self.genie_space_id, 
            "Genie",
            description="This Genie space provides enhanced context for user queries"
        )
        
    def binary_decision_function(self) -> bool:
        """
        Placeholder binary decision function.
        Returns True if random number > 0.5, False otherwise.
        
        Returns:
            bool: True or False based on random decision
        """
        return  True # random.random() > 0.5
    
    def get_fixed_response(self) -> str:
        """
        Returns a fixed response when the binary decision is False.
        
        Returns:
            str: Fixed response message
        """
        return "I'm sorry, but I'm not available right now. Please try again later."
    
    def chat_with_llm(self, user_input: str) -> str:
        """
        Chat with the LLM using Databricks endpoint, optionally enhanced with Genie context.
        
        Args:
            user_input: User's message
            
        Returns:
            str: LLM response
        """
        try:
            try:
                genie_response = self.genie_agent.invoke(user_input)
                enhanced_input = f"User Question: {user_input}\n\nGenie Context: {genie_response}\n\nPlease provide a comprehensive response based on the Genie context and user question."
                print("Genie context retrieved successfully.")
            except Exception as e:
                print(f"Warning: Genie query failed, proceeding with original input: {e}")
                enhanced_input = user_input
            
            # Create a prompt template that incorporates Genie context if available
            system_prompt = """You are a helpful AI assistant enhanced with Genie context. 
            Use the provided Genie context to give more accurate and contextual responses. 
            If the Genie context is relevant, incorporate it into your answer. 
            Always provide clear, helpful, and well-structured responses."""
            
            # Create the chain
            prompt = ChatPromptTemplate.from_messages([
                ("system", system_prompt),
                ("human", "{input}")
            ])
            
            chain = prompt | self.llm | StrOutputParser()
            
            # Get response
            response = chain.invoke({"input": enhanced_input})
            return response
            
        except Exception as e:
            return f"Error communicating with LLM: {str(e)}"
    
    def process_input(self, user_input: str) -> str:
        """
        Main processing function that implements the conditional flow using RunnableBranch.
        
        Args:
            user_input: User's message
            
        Returns:
            str: Response based on the conditional flow
        """
        # Create the conditional branch using RunnableBranch
        conditional_chain = RunnableBranch(
            # Branch 1: If binary decision is True, proceed to LLM chatbot
            (lambda x: self.binary_decision_function(), 
             RunnableLambda(lambda x: self._handle_llm_flow(x))),
            # Default branch: If binary decision is False, return fixed response
            RunnableLambda(lambda x: self._handle_fixed_response(x))
        )
        
        # Execute the conditional chain
        result = conditional_chain.invoke(user_input)
        return result
    
    def _handle_llm_flow(self, user_input: str) -> str:
        """
        Handle the LLM chatbot flow when binary decision is True.
        
        Args:
            user_input: User's message
            
        Returns:
            str: LLM response
        """
        print("Decision is True - proceeding to LLM chatbot...")
        return self.chat_with_llm(user_input)
    
    def get_decision_result(self) -> bool:
        """
        Get the current binary decision result for display purposes.
        
        Returns:
            bool: Current binary decision result
        """
        return self.binary_decision_function()
    
    def _handle_fixed_response(self, user_input: str) -> str:
        """
        Handle the fixed response flow when binary decision is False.
        
        Args:
            user_input: User's message (unused in this case)
            
        Returns:
            str: Fixed response message
        """
        print("Decision is False - returning fixed response...")
        return self.get_fixed_response()


In [0]:
# Initialize the chatbot
chatbot = ConditionalChatbot(
    databricks_host=os.getenv("DATABRICKS_HOST"),
    databricks_token=os.getenv("DATABRICKS_TOKEN"),
    genie_space_id="01f0873815fd11929465c5a98a11ff46"
)

print("This chatbot uses a binary decision function to determine whether to:")
print("1. Proceed to LLM chatbot (if decision is True)")
print("2. Return a fixed response (if decision is False)")
print("\nThe binary decision function currently uses random.random() > 0.5\n")


questions = [
    "What is Abraham Lincoln famous for?",
    "What color is the sky?",
    "Explain this dataset",
    "Show me the most valuable account in our data"
]
for q in questions:
    print(f"Question: {q}\nBot Answer:{chatbot.process_input(q)}")